# Results Comparison and Final Figures
Run this notebook after all model notebooks.

In [ ]:


import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/a213696/Group-5-ML-Project.git"
REPO_NAME = "Group-5-ML-Project"

CURRENT = Path.cwd()

if (CURRENT / "src").exists():
    PROJECT_ROOT = CURRENT

elif CURRENT.name == "notebooks" and (CURRENT.parent / "src").exists():
    PROJECT_ROOT = CURRENT.parent

else:
    BASE_DIR = Path("/content") if Path("/content").exists() else CURRENT
    PROJECT_ROOT = BASE_DIR / REPO_NAME

    if not PROJECT_ROOT.exists():
        print("Cloning repository from GitHub...")
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT_ROOT)], check=True)
    else:
        print("Repository already exists. Pulling latest version...")
        subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull"], check=False)

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

FIG_DIR = PROJECT_ROOT / "artifacts" / "figures"
TABLE_DIR = PROJECT_ROOT / "artifacts" / "tables"
MODEL_DIR = PROJECT_ROOT / "artifacts" / "models"
for d in [FIG_DIR, TABLE_DIR, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Current working directory:", Path.cwd())
print("src exists:", (PROJECT_ROOT / "src").exists())
print("notebooks exists:", (PROJECT_ROOT / "notebooks").exists())

if not (PROJECT_ROOT / "src").exists():
    raise FileNotFoundError("The src folder is missing. Upload src to GitHub or clone the full project repository, not only one notebook.")

import json, os, glob
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from src import viz



In [ ]:
def load_json(path):
    with open(path) as f: return json.load(f)
stageA = load_json(TABLE_DIR / "stageA_result.json")
stageB = load_json(TABLE_DIR / "stageB_result.json")
cnn_results = load_json(TABLE_DIR / "stageC_cnn_result.json")
stageTL = load_json(TABLE_DIR / "stageC_tl_result.json")
print("All results loaded")

In [ ]:
records = [
    {"Stage":"A", "Model":"Logistic Regression (scratch)", "Preprocessing":"flatten + z-score", "Test Acc":stageA["test_accuracy"], "Macro F1":stageA["macro_f1"], "Params":stageA.get("n_params", "N/A")},
    {"Stage":"B", "Model":"Random Forest + PCA", "Preprocessing":"flatten + PCA(150)", "Test Acc":stageB["test_accuracy"], "Macro F1":stageB["macro_f1"], "Params":"N/A"},
]
for r in cnn_results:
    records.append({"Stage":"C", "Model":r["model"], "Preprocessing":"CNN + augmentation" if "noaug" not in r["model"] else "CNN no augmentation", "Test Acc":r["test_accuracy"], "Macro F1":r["macro_f1"], "Params":r.get("n_params", "N/A")})
records.append({"Stage":"C", "Model":"MobileNetV2 Transfer", "Preprocessing":"resize + pre-trained CNN", "Test Acc":stageTL["test_accuracy"], "Macro F1":stageTL["macro_f1"], "Params":stageTL["total_params"]})
df = pd.DataFrame(records)
df["Test Acc %"] = (df["Test Acc"] * 100).round(2)
display(df)
df.to_csv(TABLE_DIR / "master_results_table.csv", index=False)

In [ ]:
plot_df = df.rename(columns={"Model":"model", "Test Acc":"accuracy"})[["model", "accuracy"]]
viz.plot_comparison_bar(plot_df, metric="accuracy")
cnn_with_aug = [r for r in cnn_results if "noaug" not in r["model"].lower()]
depths = [r["n_blocks"] for r in cnn_with_aug]
accs = [r["test_accuracy"] for r in cnn_with_aug]
viz.plot_depth_vs_accuracy(depths, accs)

In [ ]:
print("Generated figure files:")
for f in sorted(glob.glob(str(FIG_DIR / "*.png"))):
    print(os.path.basename(f))

## Final interpretation
The results show a clear progression from flattened-pixel models to spatial deep learning models. Logistic regression is useful as a baseline but is limited by its linear decision boundary. Random Forest adds non-linearity but still works on global PCA features. CNN and transfer learning perform best because they learn spatial visual features directly from the image grid.